# Jagran Lakecity University
## Faculty of Engineering & Technology (FAST)
### BTech CSE

---

## Practical Record: Assignment 2 - Module 2
### **Practical 7: Implementing Backpropagation from Scratch**

| Academic Details | Information |
| :--- | :--- |
| **Course** | Soft Computing |
| **Module** | Module II: Artificial Neural Networks |
| **Student Name** | *Shivam Indore* |
| **Enrollment No.** | *JLU2023BTCSE* |
| **Academic Session** | 2026 – 2027 |

---

### 1. Aim
To understand and implement the basic concept of Backpropagation for training a neural network.

### 2. Theory: What is Backpropagation?

Backpropagation ("backward propagation of errors") is how a **multi-layer** network learns. Gradient Descent (Experiment 6) tells us *how* to update a weight once we know its gradient — but for a network with hidden layers, we cannot directly measure the hidden layer's error. Backpropagation solves this.

#### 1. The Four Steps
1. **Forward pass:** Send input through the network to get a prediction.
2. Compute the **loss** (how wrong the prediction is).
3. **Backward pass:** Propagate the error backward from the output layer toward the input layer using the **Chain Rule**, so every weight (even hidden ones) gets a gradient.
4. Update every weight using Gradient Descent.

#### 2. The Key Equations
- **Output error:** $\delta_o = (\hat{y} - y) \odot \sigma'(z_o)$
- **Hidden error:** $\delta_h = (\delta_o W_2^T) \odot \sigma'(z_h)$
- **Weight gradients:** $\frac{\partial L}{\partial W_2} = a_h^T \delta_o$, $\frac{\partial L}{\partial W_1} = X^T \delta_h$

#### 3. Architecture
We implement a tiny network (2 inputs → 2 hidden neurons → 1 output) completely from scratch and train it on **XOR**.

### 3. Python Code: Network Setup and Helper Functions

#### What this cell does:
In this code cell, we define the sigmoid function and its derivative, initialize random weights and biases for a 2-layer network (2→2→1), and prepare the XOR training data.

In [1]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(output):
    return output * (1 - output)

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

np.random.seed(42)
input_size, hidden_size, output_size = 2, 2, 1
lr = 0.5
epochs = 5000

W1 = np.random.uniform(-1, 1, (input_size, hidden_size))
b1 = np.zeros((1, hidden_size))
W2 = np.random.uniform(-1, 1, (hidden_size, output_size))
b2 = np.zeros((1, output_size))

print(f"Architecture: {input_size} input -> {hidden_size} hidden -> {output_size} output")


Architecture: 2 input -> 2 hidden -> 1 output


### 4. Training Loop: Forward Pass + Backpropagation + Weight Update

#### What this cell does:
In this code cell, we run 5000 training iterations. Each iteration performs a forward pass, computes the error, propagates it backward using the chain rule, and updates all weights and biases.

In [2]:
for epoch in range(epochs):
    # Forward pass
    hidden_input = np.dot(X, W1) + b1
    hidden_output = sigmoid(hidden_input)
    final_input = np.dot(hidden_output, W2) + b2
    final_output = sigmoid(final_input)

    # Loss
    error = y - final_output
    loss = np.mean(error ** 2)

    # Backward pass (Backpropagation)
    d_output = error * sigmoid_derivative(final_output)
    error_hidden = d_output.dot(W2.T)
    d_hidden = error_hidden * sigmoid_derivative(hidden_output)

    # Update weights (Gradient Descent step)
    W2 += hidden_output.T.dot(d_output) * lr
    b2 += np.sum(d_output, axis=0, keepdims=True) * lr
    W1 += X.T.dot(d_hidden) * lr
    b1 += np.sum(d_hidden, axis=0, keepdims=True) * lr

    if epoch % 500 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:>5}  -  Loss: {loss:.6f}")


Epoch     0  -  Loss: 0.286300
Epoch   500  -  Loss: 0.249727
Epoch  1000  -  Loss: 0.242442
Epoch  1500  -  Loss: 0.036663
Epoch  2000  -  Loss: 0.009823
Epoch  2500  -  Loss: 0.005254
Epoch  3000  -  Loss: 0.003514
Epoch  3500  -  Loss: 0.002618
Epoch  4000  -  Loss: 0.002077
Epoch  4500  -  Loss: 0.001716
Epoch  4999  -  Loss: 0.001460


### 5. Final Predictions: Testing the Trained Network

#### What this cell does:
In this code cell, we pass all 4 XOR inputs through the trained network and print the final predictions to verify the network has learned XOR correctly.

In [3]:
print(f"{'Input':<10} | {'Predicted':<12} | {'Rounded':<8} | {'Expected'}")
print("-" * 50)
for xi, target in zip(X, y):
    hidden = sigmoid(np.dot(xi, W1) + b1)
    output = sigmoid(np.dot(hidden, W2) + b2)[0][0]
    print(f"{str(xi.astype(int)):<10} | {output:<12.4f} | {round(output):<8} | {int(target[0])}")


Input      | Predicted    | Rounded  | Expected
--------------------------------------------------
[0 0]      | 0.0369       | 0        | 0
[0 1]      | 0.9589       | 1        | 1
[1 0]      | 0.9589       | 1        | 1
[1 1]      | 0.0331       | 0        | 0


### 6. Conclusion

Backpropagation successfully distributed the output error backward, allowing even the hidden-layer weights to learn. The loss dropped from **0.29 → 0.0015** over 5000 epochs, and the network correctly classifies all 4 XOR patterns — all without any external ML library.